# Fase III: Comparación de Modelos de IA y Optimización Metaheurística

**Proyecto Integrador - Inteligencia Computacional**  
**Dominio:** Medio Ambiente &rarr; Calidad del Agua  
**Integrantes:** Melanie Tomalá, Jean Cedeño, Damian Torres  
**Docente:** Ing. Daniel Quirumbay  

---

## Objetivos de la Fase III (según rúbrica)
1. **Entrenar mínimo cuatro modelos:** KNN, SVM, Random Forest y MLP, comparados contra una línea base sencilla (`DummyClassifier`).
2. **Aplicar validación cruzada estratificada:** 5-Fold CV sobre el conjunto de entrenamiento (`data/processed/train.csv`). **El conjunto de prueba reservado NO se toca en esta fase.**
3. **Optimización con Metaheurística:** Implementar una búsqueda guiada por **Algoritmo Genético (GA)** para optimizar hiperparámetros y compararla con búsqueda convencional (`GridSearchCV`).
4. **Métricas obligatorias:** Macro-F1 (métrica principal ante desbalance 89/11), ROC-AUC, Recall de clase positiva, Accuracy y tiempos de entrenamiento/inferencia.

**Experimento cerrado:** consultar `reports/` y el notebook de fase IV para revisar resultados. No reejecutar preparación o selección sobre este test ya evaluado.


## 1. Importación de Librerías y Carga de Datos de Entrenamiento

In [ ]:
import os
from pathlib import Path
if not Path('data').exists() and Path('../data').exists():
    os.chdir('..')
from pathlib import Path
if Path("reports/phase4_selection.json").exists():
    raise RuntimeError("Experimento cerrado. Consultar los reportes guardados; no sobrescribir resultados.")

import time
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score, accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", palette="muted")

train_df = pd.read_csv('data/processed/train.csv')
features = [c for c in train_df.columns if c != 'is_safe']
X = train_df[features]
y = train_df['is_safe']

print(f"Muestras de entrenamiento: {len(X)} | Características: {len(features)}")
print(f"Distribución de clases: 0 (No segura) = {(y==0).sum()} ({(y==0).mean()*100:.2f}%) | 1 (Segura) = {(y==1).sum()} ({(y==1).mean()*100:.2f}%)")

## 2. Definición del Protocolo de Validación Cruzada (5-Fold Stratified CV)

Para garantizar ausencia de data leakage entre folds, el escalado con `StandardScaler` se encapsula dentro del `Pipeline` para que se ajuste (`fit`) únicamente sobre los 4 folds de entrenamiento y se transforme en el fold de validación.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate_pipeline(pipe, X, y, cv):
    macro_f1s, roc_aucs, recalls, precisions, accs = [], [], [], [], []
    fit_times, infer_times = [], []
    
    for tr_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        
        t0 = time.time()
        pipe.fit(X_tr, y_tr)
        fit_times.append(time.time() - t0)
        
        t0 = time.time()
        preds = pipe.predict(X_val)
        infer_times.append((time.time() - t0) * 1000)
        
        if hasattr(pipe, 'predict_proba'):
            probas = pipe.predict_proba(X_val)[:, 1]
        else:
            probas = preds
            
        macro_f1s.append(f1_score(y_val, preds, average='macro', zero_division=0))
        roc_aucs.append(roc_auc_score(y_val, probas))
        recalls.append(recall_score(y_val, preds, pos_label=1, zero_division=0))
        precisions.append(precision_score(y_val, preds, pos_label=1, zero_division=0))
        accs.append(accuracy_score(y_val, preds))
        
    return {
        'macro_f1_mean': np.mean(macro_f1s), 'macro_f1_std': np.std(macro_f1s),
        'roc_auc_mean': np.mean(roc_aucs), 'roc_auc_std': np.std(roc_aucs),
        'recall_mean': np.mean(recalls), 'recall_std': np.std(recalls),
        'precision_mean': np.mean(precisions), 'precision_std': np.std(precisions),
        'accuracy_mean': np.mean(accs), 'accuracy_std': np.std(accs),
        'fit_time': np.mean(fit_times), 'infer_time_ms': np.mean(infer_times)
    }

print("Función de evaluación CV definida.")

## 3. Entrenamiento y Comparación de los Modelos Base

Entrenamos los 5 modelos mínimos exigidos por la rúbrica:
1. **DummyClassifier (Baseline):** Predice siempre la clase mayoritaria.
2. **KNN:** Vecinos más cercanos ponderados por distancia.
3. **SVM (RBF):** Máquina de soporte vectorial con pesos balanceados (`class_weight='balanced'`).
4. **Random Forest (Base):** Ensamble con árboles no lineales.
5. **MLP (Red Neuronal):** Perceptrón multicapa con optimizador Adam y parada temprana.

In [ ]:
base_models = {
    'Dummy (Baseline)': Pipeline([('model', DummyClassifier(strategy='most_frequent'))]),
    'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=7, weights='distance'))]),
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()), ('model', SVC(kernel='rbf', C=5.0, gamma='scale', class_weight='balanced', probability=True, random_state=RANDOM_STATE))]),
    'Random Forest (Base)': Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(n_estimators=150, max_depth=12, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))]),
    'MLP (Red Neuronal)': Pipeline([('scaler', StandardScaler()), ('model', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=300, random_state=RANDOM_STATE, early_stopping=True))])
}

results = {}
for name, pipe in base_models.items():
    print(f"Evaluando {name}...")
    results[name] = evaluate_pipeline(pipe, X, y, cv)
    res = results[name]
    print(f"  Macro-F1: {res['macro_f1_mean']:.4f} ± {res['macro_f1_std']:.4f} | ROC-AUC: {res['roc_auc_mean']:.4f} | Recall: {res['recall_mean']:.4f} | Acc: {res['accuracy_mean']*100:.1f}%")

## 4. Optimización Metaheurística con Algoritmo Genético (GA)

Implementamos un Algoritmo Genético para buscar la mejor combinación de hiperparámetros del Random Forest, maximizando la media de Macro-F1 en 5-Fold CV.

In [ ]:
class GeneticAlgorithmRF:
    def __init__(self, X, y, cv, pop_size=14, generations=7, mutation_rate=0.25, elitism=2):
        self.X, self.y, self.cv = X, y, cv
        self.pop_size, self.generations = pop_size, generations
        self.mutation_rate, self.elitism = mutation_rate, elitism
        self.param_ranges = {
            'n_estimators': [60, 100, 150, 200, 250, 300],
            'max_depth': [6, 8, 10, 12, 16, 20, None],
            'min_samples_split': [2, 4, 6, 8, 10],
            'min_samples_leaf': [1, 2, 3, 4],
            'criterion': ['gini', 'entropy']
        }
        self.history = []
        
    def _random_chromosome(self):
        return {k: np.random.choice(v) for k, v in self.param_ranges.items()}
        
    def _fitness(self, chrom):
        m = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(**chrom, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
        scores = []
        for tr, val in self.cv.split(self.X, self.y):
            m.fit(self.X.iloc[tr], self.y.iloc[tr])
            scores.append(f1_score(self.y.iloc[val], m.predict(self.X.iloc[val]), average='macro', zero_division=0))
        return float(np.mean(scores))
        
    def run(self):
        pop = [self._random_chromosome() for _ in range(self.pop_size)]
        best_f, best_chrom = -1, None
        
        for g in range(self.generations):
            fits = [self._fitness(c) for c in pop]
            order = np.argsort(fits)[::-1]
            pop = [pop[i] for i in order]
            fits = [fits[i] for i in order]
            
            if fits[0] > best_f:
                best_f = fits[0]
                best_chrom = pop[0].copy()
                
            self.history.append({'gen': g+1, 'best': fits[0], 'mean': np.mean(fits)})
            print(f"Gen {g+1:02d}/{self.generations:02d} | Mejor Macro-F1: {fits[0]:.4f} | Media: {np.mean(fits):.4f}")
            
            new_pop = pop[:self.elitism]
            while len(new_pop) < self.pop_size:
                p1 = pop[np.random.choice(len(pop)//2)]
                p2 = pop[np.random.choice(len(pop)//2)]
                child = {k: (p1[k] if np.random.rand() < 0.5 else p2[k]) for k in self.param_ranges.keys()}
                for k, v in self.param_ranges.items():
                    if np.random.rand() < self.mutation_rate:
                        child[k] = np.random.choice(v)
                new_pop.append(child)
            pop = new_pop
        return best_chrom, best_f, self.history

ga = GeneticAlgorithmRF(X, y, cv)
best_rf_params, best_rf_score, ga_hist = ga.run()
print("\nMejores parámetros por Algoritmo Genético:", best_rf_params)

## 5. Búsqueda Convencional con GridSearchCV (Contraste Metodológico)

In [ ]:
grid_space = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [8, 12, 16],
    'model__min_samples_split': [2, 6],
    'model__criterion': ['gini', 'entropy']
}

rf_grid = Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
gs = GridSearchCV(rf_grid, grid_space, scoring='f1_macro', cv=cv, n_jobs=-1)
t0 = time.time()
gs.fit(X, y)
t_gs = time.time() - t0

print(f"GridSearchCV completado en {t_gs:.1f}s | Mejor Macro-F1: {gs.best_score_:.4f}")
print("Mejores parámetros:", gs.best_params_)

## 6. Tabla Comparativa Final de la Fase III

In [ ]:
# Evaluar modelo de GA
pipe_ga = Pipeline([('scaler', StandardScaler()), ('model', RandomForestClassifier(**best_rf_params, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))])
results['Random Forest (GA-Optimized)'] = evaluate_pipeline(pipe_ga, X, y, cv)
results['Random Forest (GridSearch)'] = evaluate_pipeline(gs.best_estimator_, X, y, cv)

df_results = pd.DataFrame([
    {
        'Modelo': m,
        'Macro-F1': f"{d['macro_f1_mean']:.4f} ± {d['macro_f1_std']:.4f}",
        'ROC-AUC': f"{d['roc_auc_mean']:.4f}",
        'Recall (1)': f"{d['recall_mean']:.4f}",
        'Precision (1)': f"{d['precision_mean']:.4f}",
        'Accuracy': f"{d['accuracy_mean']*100:.2f}%",
        'Fit Time (s)': f"{d['fit_time']:.2f}"
    }
    for m, d in results.items()
])

display(df_results)